In [3]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import unidecode
import matplotlib.pyplot as plt
import re

sw = stopwords.words("spanish")
snow = SnowballStemmer('spanish')

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
class Word2VecFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, sg=0):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.model = None

    def fit(self, X, y=None):
        X = [sentence.split() for sentence in X]
        self.model = Word2Vec(sentences=X, vector_size=self.vector_size, window=self.window, min_count=self.min_count, sg=self.sg,epochs=30)
        return self

    def transform(self, X):
        feature_vectors = []
        for text in X:
            vectors = [self.model.wv[word] for word in text if word in self.model.wv]
            feature_vectors.append(np.mean(vectors, axis=0))

        return np.array(feature_vectors)

def limpiar(text):
    return ' '.join([snow.stem(unidecode.unidecode(w)) for w in re.findall(r'[a-záéíóúñ]+',str(text).lower()) if (len(w)>=2) and (w not in sw)])

In [6]:
data = pd.read_excel('80s.xlsx')
data['Comentariomin'] = data['Comentariomin'].apply(limpiar)
data = pd.concat([data[data.EVALUACION==1],data[data.EVALUACION==5].sample(889,ignore_index=True),data[data.EVALUACION==3].sample(889,ignore_index=True)])

In [7]:
from sklearn.ensemble import GradientBoostingClassifier  
from sklearn.ensemble import RandomForestClassifier   
from sklearn.ensemble import ExtraTreesClassifier    
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier    
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [8]:
pipelines = []
for name_1, vectorizer in [('tf',TfidfVectorizer(use_idf=False)),('tf',TfidfVectorizer(use_idf=True)),('w2v',Word2VecFeatureExtractor())]:
    sequences = []
    for name_2, model in [('gradientboosting',GradientBoostingClassifier()),('rf',RandomForestClassifier()),('extratrees',ExtraTreesClassifier()),('lr',LogisticRegression()),('dt',DecisionTreeClassifier()),('knn',KNeighborsClassifier()),('svm',SVC()),('mlp',MLPClassifier())]:
        sequences.append(Pipeline([(name_1,vectorizer),(name_2,model)]))
    pipelines.append(sequences)

In [12]:
vector_params = {
    'tf': {
        'max_features': [20, 50, 100, 200, None],
        'ngram_range': [(1, 1), (1, 2), (1, 3)],
        'norm': ['l1', 'l2', None],
        'max_df': [0.75, 0.85]
    },
    'w2v': {
        'vector_size': [20, 50, 100, 200, 500],
        'window': [3, 5],
        'min_count': [5, 10],
        'sg': [0, 1]
    }
}

model_params = {
    "gradientboosting": {
        'n_estimators': [50, 100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
        'max_depth': [None, 10, 20, 30, 50, 100],
        'subsample': [0.6, 0.8, 1.0],
        'min_samples_split': [2, 5, 10, 30, 50],
        'min_samples_leaf': [1, 2, 4, 8, 15, 30, 50],
        'max_features': ['sqrt', 'log2', None]
    },
    "rf": {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30, 50, 100],
        'min_samples_split': [2, 5, 10, 30, 50],
        'min_samples_leaf': [1, 2, 4, 8, 15, 30, 50],
        'max_features': ['sqrt', 'log2', None],
        'bootstrap': [True, False]
    },
    "extratrees": {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30, 50, 100],
        'min_samples_split': [2, 5, 10, 30, 50],
        'min_samples_leaf': [1, 2, 4, 8, 15, 30, 50],
        'max_features': ['sqrt', 'log2', None],
        'bootstrap': [True, False]
    },
    "lr": {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'lbfgs', 'saga'],
        'penalty': ['l1', 'l2', 'elasticnet', None],
        'max_iter': [100, 200, 500, 1000]
    },
    "dt": {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [None, 10, 20, 30, 50, 100],
        'min_samples_split': [2, 5, 10, 30, 50],
        'min_samples_leaf': [1, 2, 4, 8, 15, 30, 50],
        'max_features': ['sqrt', 'log2', None]
    },
    "knn": {
        'n_neighbors': [3, 5, 10, 20],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        'p': [1, 2]
    },
    "svm": {
        'C': [0.1, 1, 10, 100],
        'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'degree': [2, 3, 4],
        'gamma': ['scale', 'auto'],
        'shrinking': [True, False]
    },
    "mlp": {
        "hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 100)],
        "activation": ["tanh", "relu"],
        "solver": ["adam", "sgd"],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate": ["constant", "adaptive"],
        "learning_rate_init": [0.001, 0.01],
        "max_iter": [200, 500],
        "batch_size": ["auto", 64, 128]
    }
}

x = {}
x.update(vector_params)
x.update(model_params)

params = {}
for key, embeds in x.items():
    for subkey, distribs in embeds.items():
        params.update({f'{key}__{subkey}':distribs})

def get_distrib(pipe):
    tmp_dict = {}
    for key, values in params.items():
        if key in pipe.get_params().keys():
            tmp_dict.update({key:values})
    return tmp_dict

In [13]:
X_T, X_t, Y_T, Y_t = train_test_split(data.Comentariomin,data.EVALUACION,test_size=0.2,random_state=42,shuffle=True,stratify=data.EVALUACION)

In [16]:
def training(pipe):
    kfold = KFold(n_splits=10, random_state=42, shuffle= True)
    T_results = []
    t_results = []
    hyperparams = []
    for subpipe in pipe:
         print(f'Training {list(subpipe.named_steps.keys())}')
         subparams = get_distrib(subpipe)
         rs = RandomizedSearchCV(
             subpipe,
             param_distributions=subparams,
             n_iter=30,
             scoring='accuracy',
             cv=5
         )
         rs.fit(X_T,Y_T)
         best = rs.best_estimator_
         
         train_acc = cross_val_score(best,X_T,Y_T,cv=kfold,scoring='accuracy')
         test_acc = accuracy_score(Y_t,best.predict(X_t))
         best_params = rs.best_params_
         
         T_results.append(train_acc)
         t_results.append(test_acc)
         hyperparams.append(best_params)
         
         print(f"""Best hyperparameters: {best_params}
Train Acc: {train_acc.mean():.4f} ({train_acc.std():.4f})
Test Acc: {test_acc}\n""")
         
    return T_results, t_results, hyperparams

def compute_boxes(T_results, t_results, title):
    models = ['gradientboosting','rf','extratrees','lr','dt','knn','svm','mlp']
    labels = [f"{tag}\n({t_results[i]*100:.2f}%)" for i,tag in enumerate(models)]
    
    fig = plt.figure(figsize=[10,7])
    fig.suptitle(title)
    ax = fig.add_subplot(111)
    plt.boxplot(T_results)
    ax.set_xticklabels(labels)
    plt.show()

In [9]:
from sklearn.preprocessing import StandardScaler
std_tf = pipelines[1]

for i,pipe in enumerate(std_tf):
    steps = pipe.steps
    steps.insert(1,('scaler',StandardScaler(with_mean=False)))
    std_tf[i].steps = steps


In [16]:
best = std_tf[2].set_params(**{'tf__norm': None, 
                      'tf__ngram_range': (1, 1), 
                      'tf__max_features': None, 
                      'tf__max_df': 0.85, 
                      'extratrees__n_estimators': 300, 
                      'extratrees__min_samples_split': 5, 
                      'extratrees__min_samples_leaf': 2, 
                      'extratrees__max_features': 'sqrt', 
                      'extratrees__max_depth': 50, 
                      'extratrees__bootstrap': False})

best.fit(X_T,Y_T)

Pipeline(steps=[('tf', TfidfVectorizer(max_df=0.85, norm=None)),
                ('scaler', StandardScaler(with_mean=False)),
                ('extratrees',
                 ExtraTreesClassifier(max_depth=50, min_samples_leaf=2,
                                      min_samples_split=5, n_estimators=300))])

In [17]:
best

Pipeline(steps=[('tf', TfidfVectorizer(max_df=0.85, norm=None)),
                ('scaler', StandardScaler(with_mean=False)),
                ('extratrees',
                 ExtraTreesClassifier(max_depth=50, min_samples_leaf=2,
                                      min_samples_split=5, n_estimators=300))])

In [33]:
work = pd.read_excel('base_limpia.xlsx')
work

,disease,title,body
0,antrax,Alarma en camal de Chincha por ántrax,"['chinch', 'ciert', 'cautel', 'much', 'respons..."
1,antrax,Detectan dos casos de ántrax en Sama Inclán,"['mari', 'llayqu', 'muert', 'dos', 'terner', '..."
2,antrax,Tacna: Senasa descarta que ántrax hallada en o...,"['miguel', 'queved', 'director', 'senas', 'con..."
3,antrax,Senasa vacuna a ovinos y caprinos en Poquera p...,"['director', 'ejecut', 'senas', 'tacn', 'mari'..."
4,antrax,Confirman la presencia de ántrax en una oveja,"['director', 'ejecut', 'servici', 'nacional', ..."
...,...,...,...
13522,zika,En sesión del Comité Distrital de Salud de Lur...,"['pm', 'oficin', 'defensori', 'puebl', 'lim', ..."
13523,zika,Piura: De seis piuranos solo uno recibe agua d...,"['nadi', 'secret', 'pesim', 'servici', 'agu', ..."
13524,zika,Dengue: ¿qué lo convierte en una enfermedad gr...,"['deng', 'enfermed', 'infecci', 'caus', 'virus..."
13525,zika,Brote de fiebre de Oropouche pone en alerta a ...,"['nuev', 'brot', 'fiebr', 'oropouch', 'brasil'..."


In [ ]:
work['body'] = work['body'].apply(lambda x: ' '.join(eval(x)))

In [35]:
work

,disease,title,body
0,antrax,Alarma en camal de Chincha por ántrax,chinch ciert cautel much respons vien tom auto...
1,antrax,Detectan dos casos de ántrax en Sama Inclán,mari llayqu muert dos terner cri hembr vac rep...
2,antrax,Tacna: Senasa descarta que ántrax hallada en o...,miguel queved director senas confirm cas antra...
3,antrax,Senasa vacuna a ovinos y caprinos en Poquera p...,director ejecut senas tacn mari bolan inform l...
4,antrax,Confirman la presencia de ántrax en una oveja,director ejecut servici nacional sanid agrari ...
...,...,...,...
13522,zika,En sesión del Comité Distrital de Salud de Lur...,pm oficin defensori puebl lim present ultim re...
13523,zika,Piura: De seis piuranos solo uno recibe agua d...,nadi secret pesim servici agu potabl ofert eps...
13524,zika,Dengue: ¿qué lo convierte en una enfermedad gr...,deng enfermed infecci caus virus pertenecient ...
13525,zika,Brote de fiebre de Oropouche pone en alerta a ...,nuev brot fiebr oropouch brasil puest alert au...


In [369]:
preds = best.predict(wig.title)

In [370]:
wig['sentiment'] = preds

In [373]:
wig['sentiment'].value_counts()

sentiment
3    8612
5    1122
1     484
Name: count, dtype: int64

In [372]:
wig.to_excel('with_sentiment.xlsx')

In [41]:
scraper = pd.read_excel('scraper2.xlsx',index_col=0)

In [68]:
was = pd.merge(left=scraper,right=work,how='right',on=['disease','title']).drop_duplicates(subset=['link'])
was = was[~(was['body_x'].isnull())][['disease','link','date','title','body_x','body_y']]

In [72]:
was.labels = ['disease','link','date','title','fullbody','token']

In [73]:
was.to_excel('base_limpia.xlsx')

In [92]:
was['link']

0        https://diariocorreo.pe/peru/alarma-en-camal-d...
2        https://diariocorreo.pe/peru/detectan-dos-caso...
4        https://rpp.pe/peru/actualidad/tacna-senasa-de...
6        https://diariocorreo.pe/peru/senasa-vacuna-a-o...
8        https://elcomercio.pe/peru/tacna/confirman-pre...
                               ...                        
20144    https://andina.pe/agencia/noticia-lucha-contra...
20145    https://andina.pe/agencia/noticia-minsa-incorp...
20146    https://diariocorreo.pe/edicion/piura/aumentan...
20150    https://noticiaspiura30.pe/piura-de-seis-piura...
20153    https://panamericana.pe/salud/419600-brote-fie...
Name: link, Length: 10218, dtype: object

In [299]:
def get_domain(link):
    try:
        domain = re.sub(r'blogs\.|www\.|\.pe|\.gob|\.com|\.org|\.edu','',re.findall(r'(?<=://)[\w\W]+?(?=/)',str(link))[0])
    except:
        domain = None
    finally:
        return domain

In [300]:
was['domain'] = was['link'].apply(lambda x: get_domain(x))

In [308]:
was.to_excel('base_limpia.xlsx')

In [360]:
sw = stopwords.words('spanish')
snow = SnowballStemmer('spanish')

# Función limpieza actualizada con stemmer. DEJA EL OBJETO EN LAS COLUMNAS COMO LISTA
def limpiar(text):
    return ' '.join([snow.stem(unidecode.unidecode(w)) for w in re.findall(r'[a-záéíóúñ]+',str(text).lower()) if (len(w)>=2) and (w not in sw)])

In [368]:
wig['title'] = wig['title'].apply(limpiar)
wig

,disease,date,title,body,tokens,domain,sentiment
0,antrax,2010-08-29,alarm camal chinch antrax,"chincha. Con cierta cautela, pero con mucha re...",chinch ciert cautel much respons vien tom auto...,diariocorreo,1
2,antrax,2012-01-12,detect dos cas antrax sam inclan,(María LLayque).- La muerte de dos terneras (c...,mari llayqu muert dos terner cri hembr vac rep...,diariocorreo,1
4,antrax,2014-02-27,tacn senas descart antrax hall ovej contagi human,"Miguel Quevedo, director del Senasa, confirmó ...",miguel queved director senas confirm cas antra...,rpp,1
6,antrax,2014-02-26,senas vacun ovin caprin poquer antrax,"El Director Ejecutivo de Senasa Tacna, Mario B...",director ejecut senas tacn mari bolan inform l...,diariocorreo,5
8,antrax,2014-02-25,confirm presenci antrax ovej,El director ejecutivo del Servicio Nacional de...,director ejecut servici nacional sanid agrari ...,elcomercio,1
...,...,...,...,...,...,...,...
20144,zika,2024-04-28,luch deng piur mins dires fortalecer uso nuev ...,\n\n\n\nLa primera etapa consiste en el recojo...,primer etap cons recoj inform permitir defin d...,andina,1
20145,zika,2024-05-08,mins incorpor fich tecnic prepar farmaceut rep...,\n\n\n\nEl “Repelente en gel” puede ser elabor...,repelent gel pued ser elabor prepar oficinal a...,andina,1
20146,zika,2024-05-27,aument quinc cas guillain barr piur,A quince aumentaron los casos del Síndrome de ...,quinc aument cas sindrom guillain barr sgb reg...,diariocorreo,1
20150,zika,2024-07-09,piur seis piuran sol recib agu maner constant,Para nadie es un secreto el pésimo servicio de...,nadi secret pesim servici agu potabl ofert eps...,noticiaspiura30,1


In [310]:
was['date']

0        Aug 29, 2010
2        Jan 12, 2012
4        Feb 27, 2014
6        Feb 26, 2014
8        Feb 25, 2014
             ...     
20144     28 abr 2024
20145      8 may 2024
20146     27 may 2024
20150      9 jul 2024
20153      hace 1 mes
Name: date, Length: 10218, dtype: object

In [311]:
'23/10/2024 '

'23/10/2024 '

In [312]:
from datetime import datetime

In [333]:
from datetime import datetime, timedelta

def parse_date(date_string, reference_date=None):
    """
    Extended parsing function to handle all cases, including:
    - Japanese relative dates like "1 個月前", "2 週前", "3 天前".
    """
    if reference_date is None:
        reference_date = datetime.today()

    # Month mapping for alternative English and Spanish abbreviations
    month_mapping_japanese = {
        'ene': 'jan', 'feb': 'feb', 'mar': 'mar', 'abr': 'apr', 'may': 'may',
        'jun': 'jun', 'jul': 'jul', 'ago': 'aug', 'sept': 'sep', 'oct': 'oct',
        'nov': 'nov', 'dic': 'dec', 'Sept': 'Sep'
    }

    try:
        # Case 1: Standard American/English date format
        return datetime.strptime(date_string, "%b %d, %Y")
    except ValueError:
        try:
            # Case 2: Standard English/Spanish abbreviated dates
            translated_date = " ".join([
                month_mapping_japanese.get(part, part) for part in date_string.split()
            ])
            return datetime.strptime(translated_date, "%d %b %Y")
        except ValueError:
            try:
                # Case 3: Japanese-style dates
                if "年" in date_string and "月" in date_string and "日" in date_string:
                    cleaned_date = date_string.replace("年", "-").replace("月", "-").replace("日", "")
                    return datetime.strptime(cleaned_date, "%Y-%m-%d")
                else:
                    raise ValueError
            except ValueError:
                try:
                    # Case 4: Relative dates in Spanish
                    if "hace" in date_string.lower():
                        if "minuto" in date_string:
                            minutes_ago = int(date_string.split()[1])
                            return reference_date - timedelta(minutes=minutes_ago)
                        elif "hora" in date_string:
                            hours_ago = int(date_string.split()[1])
                            return reference_date - timedelta(hours=hours_ago)
                        elif "día" in date_string or "dias" in date_string:
                            days_ago = int(date_string.split()[1])
                            return reference_date - timedelta(days=days_ago)
                        elif "semana" in date_string:
                            weeks_ago = int(date_string.split()[1])
                            return reference_date - timedelta(weeks=weeks_ago)
                        elif "mes" in date_string:
                            months_ago = int(date_string.split()[1])
                            return reference_date - timedelta(days=30 * months_ago)
                        else:
                            return None
                    # Case 5: Relative dates in English
                    if "ago" in date_string.lower():
                        parts = date_string.split()
                        if "weeks" in parts or "week" in parts:
                            weeks_ago = int(parts[0])
                            return reference_date - timedelta(weeks=weeks_ago)
                        elif "months" in parts or "month" in parts:
                            months_ago = int(parts[0])
                            return reference_date - timedelta(days=30 * months_ago)
                        elif "days" in parts or "day" in parts:
                            days_ago = int(parts[0])
                            return reference_date - timedelta(days=days_ago)
                        elif "hours" in parts or "hour" in parts:
                            hours_ago = int(parts[0])
                            return reference_date - timedelta(hours=hours_ago)
                        else:
                            return None
                    # Case 6: Relative dates in Japanese
                    if "個月前" in date_string:
                        months_ago = int(date_string.replace("個月前", ""))
                        return reference_date - timedelta(days=30 * months_ago)
                    elif "週前" in date_string:
                        weeks_ago = int(date_string.replace("週前", ""))
                        return reference_date - timedelta(weeks=weeks_ago)
                    elif "日前" in date_string or "天前" in date_string:
                        days_ago = int(date_string.replace("日前", "").replace("天前", ""))
                        return reference_date - timedelta(days=days_ago)
                    else:
                        return None
                except ValueError:
                    # Unable to parse
                    return None


In [340]:
was['date'] = was['date'].apply(lambda x: parse_date(x,datetime(2024, 10, 23)))

In [338]:
was.drop(columns=['dateX'],inplace=True)

In [352]:
wig = was[['disease','date','title','body_x','body_y','domain']]
wig.columns = ['disease','date','title','body','tokens','domain']
wig

,disease,date,title,body,tokens,domain
0,antrax,2010-08-29,Alarma en camal de Chincha por ántrax,"chincha. Con cierta cautela, pero con mucha re...",chinch ciert cautel much respons vien tom auto...,diariocorreo
2,antrax,2012-01-12,Detectan dos casos de ántrax en Sama Inclán,(María LLayque).- La muerte de dos terneras (c...,mari llayqu muert dos terner cri hembr vac rep...,diariocorreo
4,antrax,2014-02-27,Tacna: Senasa descarta que ántrax hallada en o...,"Miguel Quevedo, director del Senasa, confirmó ...",miguel queved director senas confirm cas antra...,rpp
6,antrax,2014-02-26,Senasa vacuna a ovinos y caprinos en Poquera p...,"El Director Ejecutivo de Senasa Tacna, Mario B...",director ejecut senas tacn mari bolan inform l...,diariocorreo
8,antrax,2014-02-25,Confirman la presencia de ántrax en una oveja,El director ejecutivo del Servicio Nacional de...,director ejecut servici nacional sanid agrari ...,elcomercio
...,...,...,...,...,...,...
20144,zika,2024-04-28,Lucha contra el dengue en Piura: Minsa y Dires...,\n\n\n\nLa primera etapa consiste en el recojo...,primer etap cons recoj inform permitir defin d...,andina
20145,zika,2024-05-08,Minsa incorpora ficha técnica del preparado fa...,\n\n\n\nEl “Repelente en gel” puede ser elabor...,repelent gel pued ser elabor prepar oficinal a...,andina
20146,zika,2024-05-27,Aumentan a quince los casos de Guillain Barré ...,A quince aumentaron los casos del Síndrome de ...,quinc aument cas sindrom guillain barr sgb reg...,diariocorreo
20150,zika,2024-07-09,Piura: De seis piuranos solo uno recibe agua d...,Para nadie es un secreto el pésimo servicio de...,nadi secret pesim servici agu potabl ofert eps...,noticiaspiura30


In [353]:
wig.to_excel('base_limpia.xlsx',index=False)

In [350]:
aa = pd.read_excel('base_limpia.xlsx',index_col=0)
aa.labels = ['disease','date','title','body','tokens','domain']
aa.to_excel('base_limpia.xlsx')
aa

,Unnamed: 0,disease,date,title,body_x,body_y,domain
0,0,antrax,2010-08-29,Alarma en camal de Chincha por ántrax,"chincha. Con cierta cautela, pero con mucha re...",chinch ciert cautel much respons vien tom auto...,diariocorreo
1,2,antrax,2012-01-12,Detectan dos casos de ántrax en Sama Inclán,(María LLayque).- La muerte de dos terneras (c...,mari llayqu muert dos terner cri hembr vac rep...,diariocorreo
2,4,antrax,2014-02-27,Tacna: Senasa descarta que ántrax hallada en o...,"Miguel Quevedo, director del Senasa, confirmó ...",miguel queved director senas confirm cas antra...,rpp
3,6,antrax,2014-02-26,Senasa vacuna a ovinos y caprinos en Poquera p...,"El Director Ejecutivo de Senasa Tacna, Mario B...",director ejecut senas tacn mari bolan inform l...,diariocorreo
4,8,antrax,2014-02-25,Confirman la presencia de ántrax en una oveja,El director ejecutivo del Servicio Nacional de...,director ejecut servici nacional sanid agrari ...,elcomercio
...,...,...,...,...,...,...,...
10213,20144,zika,2024-04-28,Lucha contra el dengue en Piura: Minsa y Dires...,\n\n\n\nLa primera etapa consiste en el recojo...,primer etap cons recoj inform permitir defin d...,andina
10214,20145,zika,2024-05-08,Minsa incorpora ficha técnica del preparado fa...,\n\n\n\nEl “Repelente en gel” puede ser elabor...,repelent gel pued ser elabor prepar oficinal a...,andina
10215,20146,zika,2024-05-27,Aumentan a quince los casos de Guillain Barré ...,A quince aumentaron los casos del Síndrome de ...,quinc aument cas sindrom guillain barr sgb reg...,diariocorreo
10216,20150,zika,2024-07-09,Piura: De seis piuranos solo uno recibe agua d...,Para nadie es un secreto el pésimo servicio de...,nadi secret pesim servici agu potabl ofert eps...,noticiaspiura30


In [356]:
wig['sentiment'] = best.predict(wig.tokens)

In [358]:
wig.to_excel('with_sentiment.xlsx')

In [379]:
wig['sentiment'].value_counts()

sentiment
3    8612
5    1122
1     484
Name: count, dtype: int64

In [378]:
wig[wig['sentiment']==3]

,disease,date,title,body,tokens,domain,sentiment
0,antrax,2010-08-29,alarm camal chinch antrax,"chincha. Con cierta cautela, pero con mucha re...",chinch ciert cautel much respons vien tom auto...,diariocorreo,3
4,antrax,2014-02-27,tacn senas descart antrax hall ovej contagi human,"Miguel Quevedo, director del Senasa, confirmó ...",miguel queved director senas confirm cas antra...,rpp,3
6,antrax,2014-02-26,senas vacun ovin caprin poquer antrax,"El Director Ejecutivo de Senasa Tacna, Mario B...",director ejecut senas tacn mari bolan inform l...,diariocorreo,3
8,antrax,2014-02-25,confirm presenci antrax ovej,El director ejecutivo del Servicio Nacional de...,director ejecut servici nacional sanid agrari ...,elcomercio,3
10,antrax,2015-02-10,moquegu bovin vacun antrax,A través del Servicio Nacional de Sanidad Agra...,trav servici nacional sanid agrari senas moque...,andina,3
...,...,...,...,...,...,...,...
20144,zika,2024-04-28,luch deng piur mins dires fortalecer uso nuev ...,\n\n\n\nLa primera etapa consiste en el recojo...,primer etap cons recoj inform permitir defin d...,andina,3
20145,zika,2024-05-08,mins incorpor fich tecnic prepar farmaceut rep...,\n\n\n\nEl “Repelente en gel” puede ser elabor...,repelent gel pued ser elabor prepar oficinal a...,andina,3
20146,zika,2024-05-27,aument quinc cas guillain barr piur,A quince aumentaron los casos del Síndrome de ...,quinc aument cas sindrom guillain barr sgb reg...,diariocorreo,3
20150,zika,2024-07-09,piur seis piuran sol recib agu maner constant,Para nadie es un secreto el pésimo servicio de...,nadi secret pesim servici agu potabl ofert eps...,noticiaspiura30,3


In [380]:
wig.to_excel('with_sentiment.xlsx')